# Дело II-05 · Знакомый почерк

**Бюро аналитических расследований, 14–15 апреля 2026 года.** Антон Карев приносит монтаж из удачных распознаваний рукописных цифр и одну агрегированную долю правильных ответов (accuracy). Вера Орлова просит восстановить процедуру: на каких классах модель ошибается, что именно перебиралось и какая конфигурация будет заморожена до финального экзамена.

**Для кого:** студент после II-04, знакомый с разбиением, базовыми моделями и кросс-валидацией.

**Результат:** исследование 8×8 изображений, сравнение k-NN и RBF SVM, небольшой объявленный `GridSearchCV`, доля правильных ответов, macro-F1, полнота (recall) по классам, многоклассовая матрица ошибок, галерея ошибок и файл блокировки модели.


## Маршрут расследования

1. Прочитать локальный CSV в `DataFrame` и проверить его SHA-256.
2. Связать таблицу 64 пикселей с изображениями `(n, 8, 8)`.
3. Зафиксировать внешнюю стратифицированную тестовую выборку.
4. Сравнить k-NN и RBF SVM только с помощью кросс-валидации обучающей выборки.
5. Выполнить небольшой, заранее объявленный перебор параметров.
6. Один раз оценить модель на тестовой выборке и исследовать ошибки по цифрам.
7. Зафиксировать модель для дела II-06.

Ориентир времени — 4–5 часов.


In [ ]:
from __future__ import annotations

import hashlib
import random
import urllib.request
import zipfile
from pathlib import Path

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
NOTEBOOK_VARIANT = "learner"
CASE_SLUG = "case-05"
ARCHIVE_NAME = "part-2-case-05.zip"
COURSE_SITE = "https://mkuziuk.github.io/python-tutorial"
IN_COLAB = False

# При локальном запуске используем файлы из каталога дела; в Colab скачиваем архив и проверяем его SHA-256.
try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def find_local_case() -> Path | None:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / f"{CASE_SLUG}.ipynb").is_file()
        ):
            return candidate
        nested = candidate / "projects" / "part-2" / CASE_SLUG
        if (nested / "README.md").exists():
            return nested
    return None

def download_colab_case() -> Path:
    destination = Path("/content") / f"python-tutorial-{CASE_SLUG}"
    destination.mkdir(parents=True, exist_ok=True)
    archive_path = destination / ARCHIVE_NAME
    archive_url = f"{COURSE_SITE}/downloads/{ARCHIVE_NAME}"
    checksum_url = f"{archive_url}.sha256"

    urllib.request.urlretrieve(archive_url, archive_path)
    # Сравниваем SHA-256 архива с опубликованной контрольной суммой перед распаковкой.
    with urllib.request.urlopen(checksum_url) as response:
        expected = response.read().decode("utf-8").split()[0].lower()
    actual = sha256_file(archive_path)
    if actual != expected:
        raise RuntimeError(f"SHA-256 архива не совпал: {actual} != {expected}")

    unpacked = destination / "unpacked"
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(unpacked)
    matches = sorted(unpacked.rglob(f"{CASE_SLUG}.ipynb"))
    if not matches:
        raise FileNotFoundError(f"В архиве нет {CASE_SLUG}.ipynb")
    return matches[0].parent

# DATA_DIR и ARTIFACTS_DIR строятся от найденного каталога дела, поэтому текущая папка не влияет на пути.
CASE_DIR = find_local_case()
if CASE_DIR is None and IN_COLAB:
    CASE_DIR = download_colab_case()
if CASE_DIR is None:
    raise FileNotFoundError(
        f"Не найден каталог {CASE_SLUG}. Запустите тетрадь из каталога дела."
    )

DATA_DIR = CASE_DIR / "data"
ARTIFACTS_DIR = CASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
print(f"Среда: {'Colab' if IN_COLAB else 'local'} | дело: {CASE_DIR}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Источник, DataFrame и checksum

В архив вложен замороженный `data/digits.csv`: 1 797 строк, 64 пиксельных столбца со значениями 0–16 и целевой столбец `digit`. Сеть не нужна. `SOURCE.md`, `LICENSE.txt` и `dataset_manifest.json` описывают происхождение и SHA-256 файла.

Сначала прочитаем CSV в `DataFrame`. Только после проверки столбцов преобразуем пиксели в массив `(n, 64)` и изображения `(n, 8, 8)`: так переход от таблицы к формату модели остаётся видимым.


In [ ]:
manifest = json.loads((DATA_DIR / "dataset_manifest.json").read_text(encoding="utf-8"))
data_path = DATA_DIR / manifest["filename"]
actual_sha256 = sha256_file(data_path)
assert actual_sha256 == manifest["sha256"], "SHA-256 digits.csv не совпадает с карточкой"

digits_frame = pd.read_csv(data_path)
pixel_columns = list(manifest["feature_names"])
assert digits_frame.shape == (manifest["rows"], manifest["columns"])
assert digits_frame[pixel_columns].to_numpy().min() >= manifest["pixel_range"][0]
assert digits_frame[pixel_columns].to_numpy().max() <= manifest["pixel_range"][1]

X = digits_frame[pixel_columns].to_numpy(dtype=np.float64)
y = digits_frame[manifest["target"]].to_numpy(dtype=np.int64)
images = X.reshape(-1, *manifest["image_shape"])
print(f"Файл: {data_path.name} | SHA-256: {actual_sha256[:12]}…")
display(digits_frame.head())
print(f"X={X.shape}, images={images.shape}, y={y.shape}, classes={np.unique(y)}")


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for digit, ax in enumerate(axes.ravel()):
    sample_index = int(np.flatnonzero(y == digit)[0])
    ax.imshow(images[sample_index], cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"Класс {digit}")
    ax.axis("off")
fig.suptitle("По одному исходному примеру каждого класса")
plt.tight_layout()
plt.show()


In [ ]:
class_counts = pd.Series(y, name="digit").value_counts().sort_index()
pixel_summary = pd.Series(X.ravel()).describe(percentiles=[0.25, 0.5, 0.75])
display(pd.DataFrame({"count": class_counts}))
display(pixel_summary.to_frame("pixel_value").round(2))
print("Проверка разворота:", np.array_equal(X[0].reshape(8, 8), images[0]))


## 2. Внешняя тестовая выборка

Тестовая выборка остаётся изолированной до окончания выбора модели. `stratify=y` сохраняет доли десяти классов. В переборе параметров участвует только обучающая выборка.

**Упражнение:** добавьте стратификацию и проверьте, что обучающая и тестовая выборки не пересекаются по индексам.


In [ ]:
all_indices = np.arange(len(y))
# TODO: добавьте stratify=y и объясните, зачем он нужен.
train_indices, test_indices = train_test_split(
    all_indices, test_size=0.25, random_state=RANDOM_STATE
)
X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]
print(f"Временное разбиение: обучение={len(train_indices)}, тест={len(test_indices)}")


## 3. k-NN против RBF SVM

k-NN классифицирует по близким обучающим примерам; RBF SVM строит нелинейную границу по сходству. Обоим помогает единый масштаб признаков, поэтому `StandardScaler` находится внутри конвейера и обучается отдельно в каждой части кросс-валидации.

Macro-F1 сначала считает F1 каждого класса, затем усредняет классы поровну. Это мешает большому или лёгкому классу скрыть слабый.


In [ ]:
# Базовая модель задаёт нижнюю планку на той же зафиксированной тестовой выборке.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_macro_f1 = f1_score(y_test, baseline_predictions, average="macro")
print(f"Dummy macro-F1: {baseline_macro_f1:.3f}")


In [ ]:
candidates = {
    "knn_5": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "rbf_svm": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
}
# Кандидаты сравниваются на одинаковых стратифицированных частях обучающей выборки.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# TODO: вызовите cross_validate для каждой модели и соберите обе метрики.
cv_comparison = pd.DataFrame(
    columns=["accuracy_mean", "macro_f1_mean", "macro_f1_std"]
)
print("Таблица CV пока пуста — выполните TODO.")


### Маленький объявленный поиск

До запуска фиксируем четыре комбинации: `C ∈ {2, 10}` и `gamma ∈ {scale, 0.001}`. Метрика выбора — macro-F1, части — те же три стратифицированные части, `n_jobs=1`. Расширять сетку после взгляда на тестовую выборку нельзя.


In [ ]:
svm_pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
parameter_grid = {"svc__C": [2, 10], "svc__gamma": ["scale", 0.001]}
# TODO: создайте GridSearchCV только на X_train/y_train.
grid = None
best_model = clone(svm_pipeline).fit(X_train, y_train)
locked_config = {
    "pipeline": "StandardScaler -> SVC",
    "kernel": "rbf",
    "C": "TODO",
    "gamma": "TODO",
    "selection_metric": "macro_f1",
    "cv": "TODO",
}
grid_table = pd.DataFrame(columns=[
    "param_svc__C", "param_svc__gamma", "mean_test_score", "std_test_score"
])
print("Временная SVM обучена; завершите grid и lock.")


## 4. Оцениваем модель на тестовой выборке один раз

Конфигурация уже зафиксирована. Теперь получаем accuracy, macro-F1 и recall каждого класса. Тестовая выборка не участвует в новой настройке.


In [ ]:
test_predictions = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(y_test, test_predictions, average="macro")
per_class_recall = pd.Series(
    recall_score(y_test, test_predictions, average=None),
    index=np.arange(10),
    name="recall",
)
print(f"Holdout accuracy={test_accuracy:.3f}; macro-F1={test_macro_f1:.3f}")
display(per_class_recall.to_frame().round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=np.arange(10), cmap="Blues", ax=ax,
    colorbar=False,
)
ax.set_title("Какие цифры модель путает")
ax.grid(False)
plt.show()


## 5. Галерея ошибок вместо монтажа успехов

Ошибки сортируем воспроизводимо по индексу тестовой выборки. Галерея не объясняет причину автоматически: она помогает сформулировать проверяемую гипотезу о форме штриха или паре классов.


In [ ]:
# shown содержит первые 12 ошибок в порядке индексов тестовой выборки.
error_positions = np.flatnonzero(test_predictions != y_test)
shown = error_positions[: min(12, len(error_positions))]
if len(shown):
    fig, axes = plt.subplots(3, 4, figsize=(8, 6))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, local_position in zip(axes.ravel(), shown, strict=False):
        source_index = test_indices[local_position]
        ax.imshow(images[source_index], cmap="gray_r", vmin=0, vmax=16)
        ax.set_title(f"истина {y_test[local_position]} → {test_predictions[local_position]}")
        ax.axis("off")
    fig.suptitle("Первые воспроизводимые ошибки тестовой выборки")
    plt.tight_layout()
    plt.show()
print(f"Ошибок в тестовой выборке: {len(error_positions)} из {len(y_test)}")


In [ ]:
class_report = pd.DataFrame(
    classification_report(
        y_test, test_predictions, labels=np.arange(10),
        output_dict=True, zero_division=0,
    )
).T
display(class_report.loc[[str(i) for i in range(10)], ["precision", "recall", "f1-score", "support"]].round(3))


### Упражнение: заморозьте модель и вывод

Запишите `model_lock.json` до перехода к II-06. В аудиторской записке не называйте ошибку «почерком конкретного автора»: Digits не содержит такой идентификации.


In [ ]:
# TODO: заполните конфигурацию только после train CV/grid, затем вывод — по holdout.
lock_payload = {
    **locked_config,
    "random_state": RANDOM_STATE,
    "outer_holdout_opened_once": True,
    "holdout_accuracy": float(test_accuracy),
    "holdout_macro_f1": float(test_macro_f1),
    "weakest_recall_class": int(per_class_recall.idxmin()),
    "weakest_recall": float(per_class_recall.min()),
}
memo = {
    "established_fact": "TODO",
    "supported_interpretation": "TODO",
    "not_proven": "TODO",
    "limitations": "TODO",
    "recommended_action": "TODO",
}
(ARTIFACTS_DIR / "model_lock.json").write_text(
    json.dumps(lock_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Черновик lock записан; заполните TODO.")


In [ ]:
if NOTEBOOK_VARIANT == "solution":
    assert set(train_indices).isdisjoint(test_indices)
    assert grid.best_params_ == {"svc__C": 2, "svc__gamma": "scale"}
    assert test_macro_f1 > 0.95
    assert test_macro_f1 > baseline_macro_f1 + 0.50
    assert len(error_positions) >= 1
    assert set(memo) == {
        "established_fact", "supported_interpretation", "not_proven",
        "limitations", "recommended_action"
    }
    print("Проверки решения II-05 пройдены.")
else:
    print("Учебный вариант: строгие проверки включатся после сверки с решением.")


## Дело закрыто

Выполните **Restart Kernel → Run All**. Проверьте `check_result.md`: контрольную сумму, стратифицированное разбиение, перебор параметров только на обучающей выборке, macro-F1 на тестовой выборке, recall каждого класса, матрицу ошибок, галерею ошибок и `artifacts/model_lock.json`.

**Типичная ошибка:** расширять сетку параметров после просмотра тестовой выборки. Тогда тестовая выборка неявно становится проверочной.

**Расширение:** сравните нормированную матрицу ошибок с абсолютной и объясните, на какой вопрос отвечает каждая.
